# Data Engineering Interview Prep: Lesson 1
## Topic: Easy SQL - Twitter's "Histogram of Tweets"

### **The Problem**
Twitter wants to understand user engagement. Write a query to obtain a **histogram of tweets** posted per user in 2022. 
* **X-Axis (Bucket):** The number of tweets posted by a user.
* **Y-Axis (Count):** The number of users who fall into that specific tweet bucket.

#### **`tweets` Table Schema**
| Column Name | Type | Description |
| :--- | :--- | :--- |
| `tweet_id` | integer | Unique ID for each tweet |
| `user_id` | integer | Unique ID for the user |
| `msg` | string | Content of the tweet |
| `tweet_date` | timestamp | Date and time the tweet was posted |

---

### **The Logic (Two-Step Aggregation)**
To create a histogram, we must perform a "count of counts":
1. **Inner Query (Step 1):** Count how many tweets each `user_id` has in the year 2022. This creates our "buckets".
2. **Outer Query (Step 2):** Group by those buckets and count how many users are in each one.

---

### **The Solution (PostgreSQL)**
```sql
WITH tweet_counts AS (
  SELECT 
    user_id, 
    COUNT(tweet_id) AS tweet_bucket
  FROM tweets
  WHERE tweet_date >= '2022-01-01' 
    AND tweet_date < '2023-01-01'
  GROUP BY user_id
)

SELECT 
  tweet_bucket, 
  COUNT(user_id) AS users_num
FROM tweet_counts
GROUP BY tweet_bucket
ORDER BY tweet_bucket;

# Senior Data Engineer Perspective

## 1. Scalability & Partitioning
- In a distributed environment (e.g., **Databricks/Spark**), a table like `tweets` would be **partitioned by date**.  
- Using a **range filter on `tweet_date`** is critical for **partition pruning**, which avoids scanning the entire historical dataset.

## 2. Handling Data Skew
- The first aggregation `GROUP BY user_id` can cause **data skew** if a small number of "Power Users" have millions of tweets.  
- In Spark, we might need to:
  - Use **salted keys** to distribute heavy users across partitions.  
  - **Broadcast smaller metadata tables** to handle shuffles more efficiently.  
  - Consider **adaptive query execution (AQE)** for automatic skew mitigation.

## 3. Performance Tip
- `COUNT(tweet_id)` is standard, but `COUNT(1)` or `COUNT(*)` is often **slightly faster** in some engines.  
- Reason: they don’t need to check for nulls in a specific column.  
- Since `tweet_id` is typically a **primary key** and non-nullable, the difference is minimal but worth noting for performance tuning.


# Data Engineering Interview Prep: Lesson 2
## Topic: Easy SQL - LinkedIn's "Data Science Skills"

### **The Problem**
Given a table of candidates and their skills, write a query to list the **candidates who possess all of the following skills**: Python, Tableau, and PostgreSQL.

* Output the `candidate_id` in ascending order.

#### **`candidates` Table Schema**
| Column Name | Type | Description |
| :--- | :--- | :--- |
| `candidate_id` | integer | Unique ID for the candidate |
| `skill` | varchar | The name of the skill (e.g., 'Python', 'Java') |

---

### **The Logic (The "Set" Approach)**
This is a "Relational Division" problem. We aren't just looking for one skill; we are looking for a **specific set**.
1. **Filter:** First, filter the table to only rows containing the three required skills.
2. **Group:** Group by the `candidate_id`.
3. **Count & Validate:** Count how many distinct required skills each candidate has. If that count is **3**, they have everything we need.

---

### **The Solution (PostgreSQL)**
```sql
SELECT candidate_id
FROM candidates
WHERE skill IN ('Python', 'Tableau', 'PostgreSQL')
GROUP BY candidate_id
HAVING COUNT(skill) = 3
ORDER BY candidate_id ASC;

# Senior Data Engineer Perspective

## 1. Predicate Pushdown
- In large-scale systems (like **Bluelake PSS**), the `WHERE skill IN (...)` clause is vital.  
- It allows the storage engine to **skip reading rows** for `'Java'` or `'C++'`, significantly reducing the amount of data sent to the **Shuffle phase** during the `GROUP BY`.

## 2. Handling Duplicates
- If the source system isn't perfectly clean, a candidate could have `'Python'` listed twice.  
- Using `COUNT(skill)` would fail in this scenario.  
- A more robust approach is to use:  
```sql
  COUNT(DISTINCT skill) = 3
```
This ensures accuracy regardless of data quality.

## 3. Performance vs. Flexibility
- If the list of required skills grew to 50 items, the IN clause remains efficient.

- However, if the Required Skills were stored in another table, you would use:

An INNER JOIN

Or an INTERSECT operator
to find the commonality between candidate skills and required skills.

# Data Engineering SQL Lessons

## Lesson 3: Facebook - "Page With No Likes"

### The Problem
Write a query to return the IDs of the Facebook pages that have zero likes. The output should be sorted in ascending order.  
**Tables:**  
- `pages (page_id, page_name)`  
- `page_likes (user_id, page_id, liked_date)`

### The Logic (The Anti-Join)
We need to find records in the `pages` table that do *not* exist in the `page_likes` table.

### The Solution (PostgreSQL)
```sql
SELECT p.page_id
FROM pages p
LEFT JOIN page_likes pl
  ON p.page_id = pl.page_id
WHERE pl.page_id IS NULL
ORDER BY p.page_id ASC;
```

### Senior Data Engineer Perspective
- **LEFT JOIN vs. NOT IN:**  
  While `WHERE page_id NOT IN (SELECT page_id...)` works, it is dangerous.  
  If the subquery returns even a single `NULL`, the entire `NOT IN` clause evaluates to unknown, returning zero rows.

- **Recommended Approach:**  
  Use `LEFT JOIN ... IS NULL` or `NOT EXISTS` — safer and better optimized.

### PySpark Equivalent
```python
pages_df.join(likes_df, "page_id", "left_anti")
```

---

## Lesson 4: Tesla - "Unfinished Parts"

### The Problem
Tesla needs to find parts that have begun the assembly process but are not yet finished.

**Table:**  
- `parts_assembly (part, finish_date, assembly_step)`

### The Logic (Null Handling)
An unfinished part is simply a record where the `finish_date` is missing.

### The Solution (PostgreSQL)
```sql
SELECT part, assembly_step
FROM parts_assembly
WHERE finish_date IS NULL;
```

### Senior Data Engineer Perspective
- **The Cost of NULLs:**  
  In distributed file formats like Parquet, checking for `NULL` is very fast due to metadata (null counts).

- **Optimization Strategy:**  
  Instead of scanning large datasets, consider adding a flag column like:
  - `is_finished = TRUE/FALSE`
  - Partitioning based on this flag improves query performance.

---

## Lesson 5: New York Times - "Laptop vs. Mobile Viewership"

### The Problem
Calculate total viewership for:
- Laptop
- Mobile (Tablet + Phone)

**Table:**  
- `viewership (user_id, device_type, view_time)`

### The Logic (Conditional Aggregation)
Use `CASE` statements inside aggregation functions to compute multiple metrics in a single scan.

### The Solution (PostgreSQL)
```sql
SELECT 
  SUM(CASE WHEN device_type = 'laptop' THEN 1 ELSE 0 END) AS laptop_views,
  SUM(CASE WHEN device_type IN ('tablet', 'phone') THEN 1 ELSE 0 END) AS mobile_views
FROM viewership;
```

### Senior Data Engineer Perspective
- **One-Pass Aggregation:**  
  Ensures the table is scanned only once → major performance benefit.

### PySpark Equivalent
```python
from pyspark.sql import functions as F

df.select(
    F.sum(F.when(F.col("device_type") == "laptop", 1).otherwise(0)).alias("laptop_views"),
    F.sum(F.when(F.col("device_type").isin("tablet", "phone"), 1).otherwise(0)).alias("mobile_views")
)
```

- **Why Not Pivot?**  
  - `pivot()` triggers a shuffle and requires computing distinct values.
  - `when + sum` is more efficient for fixed categories.



# Data Engineering SQL Lessons (Part 2)

## Lesson 6: Facebook - "Average Post Hiatus (Part 1)"

### The Problem
Find the number of days between each userâ€™s first and last post in 2021.  
Include only users with at least 2 posts.

**Table:**  
- `posts (user_id, post_id, post_date, post_content)`

### The Logic (Date Math & HAVING Clause)
1. Filter posts for 2021  
2. Group by user  
3. Use HAVING to ensure at least 2 posts  
4. Compute MAX(date) - MIN(date)

### Solution (PostgreSQL)
```sql
SELECT 
  user_id, 
  EXTRACT(DAY FROM MAX(post_date) - MIN(post_date)) AS days_between
FROM posts
WHERE EXTRACT(YEAR FROM post_date) = 2021
GROUP BY user_id
HAVING COUNT(post_id) >= 2;
```

### Senior Data Engineer Perspective
- Avoid non-sargable conditions:
```sql
WHERE post_date >= '2021-01-01' 
  AND post_date < '2022-01-01'
```

### PySpark Equivalent
```python
from pyspark.sql import functions as F

df.groupBy("user_id").agg(
    F.datediff(F.max("post_date"), F.min("post_date")).alias("days_between"),
    F.count("post_id").alias("cnt")
).filter("cnt >= 2")
```

---

## Lesson 7: Microsoft - "Teams Power Users"

### Problem
Find top 2 users with highest messages in Aug 2022.

**Table:**  
- `messages`

### Solution
```sql
SELECT 
  sender_id,
  COUNT(message_id) AS message_count
FROM messages
WHERE sent_date >= '2022-08-01' 
  AND sent_date < '2022-09-01'
GROUP BY sender_id
ORDER BY message_count DESC
LIMIT 2;
```

### Senior Perspective
- Top-N optimized using Top-K algorithms  
- Avoid full global sort in distributed systems  

---

## Lesson 8: LinkedIn - "Duplicate Job Listings"

### Problem
Count companies with duplicate jobs.

### Solution
```sql
WITH duplicates AS (
  SELECT company_id
  FROM job_listings
  GROUP BY company_id, title, description
  HAVING COUNT(job_id) > 1
)

SELECT COUNT(DISTINCT company_id) AS duplicate_companies
FROM duplicates;
```

### Senior Perspective
- Use hashing (MD5/SHA) for large text columns  
- Reduces shuffle and improves performance  

---

## Lesson 9: Robinhood - "Cities With Completed Trades"

### Problem
Top 3 cities with most completed trades.

### Solution
```sql
SELECT 
  u.city, 
  COUNT(t.order_id) AS total_orders
FROM trades t
JOIN users u 
  ON t.user_id = u.user_id
WHERE t.status = 'Completed'
GROUP BY u.city
ORDER BY total_orders DESC
LIMIT 3;
```

### Senior Perspective
- Use Broadcast Join for small dimension tables  
- Avoid shuffling large fact tables  



# Data Engineering SQL Lessons (Part 3)

## Lesson 10: Amazon - "Average Review Ratings"

### Problem
Find average star rating per product per month.

**Table:**  
- `reviews (review_id, user_id, submit_date, product_id, stars)`

### Solution
```sql
SELECT 
  EXTRACT(MONTH FROM submit_date) AS mth,
  product_id,
  ROUND(AVG(stars), 2) AS avg_stars
FROM reviews
GROUP BY 
  EXTRACT(MONTH FROM submit_date), 
  product_id
ORDER BY 
  mth, 
  product_id;
```

### Senior Perspective
- Use pre-aggregated tables / materialized views for dashboards  
- Avoid repeated heavy scans  

---

## Lesson 11: FAANG - "Well Paid Employees"

### Problem
Find employees earning more than their managers.

### Solution
```sql
SELECT e.name AS employee_name
FROM employee e
JOIN employee m 
  ON e.manager_id = m.employee_id
WHERE e.salary > m.salary;
```

### Senior Perspective
- Self-joins are expensive at scale  
- Use hierarchy flattening or graph processing (GraphFrames)  

---

## Lesson 12: PayPal - "Final Account Balance"

### Problem
Compute final balance per account.

### Solution
```sql
SELECT 
  account_id,
  SUM(
    CASE 
      WHEN transaction_type = 'Deposit' THEN amount 
      WHEN transaction_type = 'Withdrawal' THEN -amount 
      ELSE 0 
    END
  ) AS final_balance
FROM transactions
GROUP BY account_id;
```

### Senior Perspective
- Ensure idempotency  
- Avoid duplicate processing in pipelines  

---

## Lesson 13: Facebook - "App Click-through Rate (CTR)"

### Problem
Calculate CTR per app for 2022.

### Solution
```sql
SELECT 
  app_id,
  ROUND(
    100.0 * SUM(CASE WHEN event_type = 'click' THEN 1 ELSE 0 END) / 
    NULLIF(SUM(CASE WHEN event_type = 'impression' THEN 1 ELSE 0 END), 0)
  , 2) AS ctr
FROM events
WHERE timestamp >= '2022-01-01' 
  AND timestamp < '2023-01-01'
GROUP BY app_id;
```

### Senior Perspective
- Use NULLIF to prevent division-by-zero failures  
- Critical for production stability  



# Data Engineering SQL Lessons (Part 4)

## Lesson 14: TikTok - "Second Day Confirmation"

### Problem
Find users who confirmed signup on the second day.

### Solution
```sql
SELECT e.user_id
FROM emails e
JOIN texts t 
  ON e.email_id = t.email_id
WHERE t.signup_action = 'Confirmed'
  AND t.action_date = e.signup_date + INTERVAL '1 day';
```

### Senior Perspective
- Handle timezones carefully (use UTC)
- PySpark equivalent: date_add()

---

## Lesson 15: IBM - "Product Analytics"

### Problem
Histogram of queries per employee (including 0)

### Solution
```sql
WITH query_counts AS (
  SELECT 
    e.emp_id,
    COUNT(q.query_id) AS query_count
  FROM employees e
  LEFT JOIN queries q 
    ON e.emp_id = q.emp_id 
    AND q.query_date >= '2023-07-01' 
    AND q.query_date < '2023-10-01'
  GROUP BY e.emp_id
)

SELECT 
  query_count AS unique_queries,
  COUNT(emp_id) AS employee_count
FROM query_counts
GROUP BY query_count
ORDER BY unique_queries;
```

### Senior Perspective
- Always apply filters in JOIN condition for LEFT JOIN
- Avoid turning LEFT JOIN into INNER JOIN

---

## Lesson 16: JPMorgan - "Cards Issued Difference"

### Problem
Find difference between max and min issued cards per card type

### Solution
```sql
SELECT 
  card_name, 
  MAX(issued_amount) - MIN(issued_amount) AS difference
FROM monthly_cards_issued
GROUP BY card_name
ORDER BY difference DESC;
```

### Senior Perspective
- Columnar formats (Parquet) optimize MIN/MAX via metadata

---

## Lesson 17: Alibaba - "Compressed Mean"

### Problem
Calculate weighted mean of items per order

### Solution
```sql
SELECT 
  ROUND(
    SUM(item_count::DECIMAL * order_occurrences) 
    / SUM(order_occurrences)
  , 1) AS mean
FROM items_per_order;
```

### Senior Perspective
- Avoid integer division errors using casting
- Example of pre-aggregated fact tables

